# CNN-Regime Selective Trajectory Calibration

## 1. Purpose And Model

This notebook keeps the successful selective trajectory-context representation and runs a narrow calibration pass around its best rows. The prior run promoted a logit-switch floor 0.15 candidate with Huber alpha 0.05 and shrink 0.40, while XGBoost-weighted rows were stress-strong. This audit tightens those local calibration settings.

## 2. Setup

In [1]:
"Configure imports, paths, and deterministic settings."

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

pd.set_option("display.max_columns", 180)
pd.set_option("display.width", 220)

ROOT = Path.cwd()
if not (ROOT / "context").exists():
    ROOT = next(parent for parent in ROOT.parents if (parent / "context").exists())

RUN_DATE = "2026-07-01"
RUN_TAG = "cnn_regime_selective_calibration"
RUN_ID = f"{RUN_DATE}__{RUN_TAG}"
SEED = 20260630
FORMATION_COLS = ["ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA"]
PRIMARY_FOLD = "strict_hybrid_fold"
STRESS_FOLD = "analog_distance_fold"
MAX_TARGET_ROWS_PER_WELL = 120
ENSEMBLE_SEEDS = [20260630, 20260631]

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
TABLE_DIR = ROOT / "artifacts" / "tables"
REPORT_DIR = ROOT / "artifacts" / "reports"
RUN_DIR = ROOT / "artifacts" / "logs" / "runs" / RUN_ID
for directory in [TABLE_DIR, REPORT_DIR, RUN_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "run_id": RUN_ID,
    "seed": SEED,
    "formation_cols": FORMATION_COLS,
    "primary_fold": PRIMARY_FOLD,
    "stress_fold": STRESS_FOLD,
    "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL,
    "ensemble_seeds": ENSEMBLE_SEEDS,
    "current_public_best_rmse": 14.472,
    "best_xgb_known_state_local_rmse": 13.4334,
    "best_xgb_known_state_stress_rmse": 13.7533,
    "promotion_margin": 0.05,
    "baseline_regime_mean_rmse": 13.092783707491265,
    "baseline_regime_mean_stress_rmse": 13.400574608144202,
}
with (RUN_DIR / "config.json").open("w", encoding="utf-8") as handle:
    json.dump(CONFIG, handle, indent=2)

print(ROOT)
print(RUN_DIR)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\logs\runs\2026-07-01__cnn_regime_selective_calibration


## 3. Data And Fold-Safe Pseudo Tops

In [2]:
"Build sampled target-zone rows with legal predictors and direct top-elevation labels."

folds = pd.read_csv(TABLE_DIR / "validation__strict_cv_fold_assignments__2026-06-26.csv")
descriptors = pd.read_csv(TABLE_DIR / "model__typewell_alignment_feature_descriptors__2026-06-23.csv")
well_meta = folds.merge(descriptors, on="wellname", how="left")
well_meta = well_meta.drop_duplicates("wellname").reset_index(drop=True)

DESCRIPTOR_COLS = [
    "known_rows", "known_tvt_span", "known_md_span", "known_gr_mean_last300", "known_gr_std_last300", "known_gr_slope_last300",
    "xy_span_known", "z_drop_known", "typewell_corr_last300", "typewell_mae_last300", "typewell_best_lag_corr",
    "typewell_best_lag_tvt", "typewell_best_lag_mae", "selected_analog_distance", "extrapolation_stress_score",
    "analog_stress_score", "combined_stress_score", "descriptor_cluster",
]


def safe_divide(num, den):
    return num / (den + 1e-9)


def load_typewell(directory, wellname):
    path = Path(directory) / f"{wellname}__typewell.csv"
    if not path.exists():
        return pd.DataFrame(columns=["TVT", "GR", "Geology"])
    return pd.read_csv(path)


def interp_typewell_gr(typewell, tvt_values):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(len(tvt_values), np.nan)
    clean = typewell[["TVT", "GR"]].dropna().sort_values("TVT")
    if len(clean) < 5:
        return np.full(len(tvt_values), np.nan)
    return np.interp(np.asarray(tvt_values, dtype=float), clean["TVT"].to_numpy(), clean["GR"].to_numpy(), left=np.nan, right=np.nan)



def robust_standardize_array(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros_like(arr, dtype=float)
    med = float(np.nanmedian(arr[finite]))
    mad = float(np.nanmedian(np.abs(arr[finite] - med)))
    scale = 1.4826 * mad if mad > 1e-9 else float(np.nanstd(arr[finite]))
    if not np.isfinite(scale) or scale <= 1e-9:
        scale = 1.0
    return np.clip((np.where(finite, arr, med) - med) / scale, -8.0, 8.0)


def compact_dtw_distance(left, right):
    x = robust_standardize_array(left)
    y = robust_standardize_array(right)
    if len(x) == 0 or len(y) == 0:
        return 0.0
    cost = np.full((len(x) + 1, len(y) + 1), np.inf)
    cost[0, 0] = 0.0
    for i in range(1, len(x) + 1):
        xi = x[i - 1]
        for j in range(1, len(y) + 1):
            step = abs(xi - y[j - 1])
            cost[i, j] = step + min(cost[i - 1, j], cost[i, j - 1], cost[i - 1, j - 1])
    return float(cost[len(x), len(y)] / max(len(x), len(y)))


def safe_window_corr(left, right):
    x = robust_standardize_array(left)
    y = robust_standardize_array(right)
    n = min(len(x), len(y))
    if n < 5:
        return 0.0
    x = x[:n]
    y = y[:n]
    if float(np.std(x)) < 1e-9 or float(np.std(y)) < 1e-9:
        return 0.0
    corr = float(np.corrcoef(x, y)[0, 1])
    return corr if np.isfinite(corr) else 0.0


def local_slope(values):
    arr = np.asarray(values, dtype=float)
    finite = np.isfinite(arr)
    if finite.sum() < 5:
        return 0.0
    x = np.arange(len(arr), dtype=float)[finite]
    y = arr[finite]
    return float(np.polyfit(x, y, 1)[0])


def local_gr_window(gr_values, center_idx, radius=16):
    start = max(0, int(center_idx) - radius)
    stop = min(len(gr_values), int(center_idx) + radius + 1)
    return np.asarray(gr_values[start:stop], dtype=float)


def typewell_window_gr(typewell, center_tvt, radius=40.0, length=33):
    if typewell.empty or not {"TVT", "GR"}.issubset(typewell.columns):
        return np.full(length, np.nan)
    grid = np.linspace(float(center_tvt) - radius, float(center_tvt) + radius, length)
    return interp_typewell_gr(typewell, grid)


def build_target_rows(csv_path, meta_row):
    df = pd.read_csv(csv_path).sort_values("MD").reset_index(drop=True)
    known_mask = df["TVT_input"].notna()
    if known_mask.sum() == 0 or (~known_mask).sum() == 0:
        return pd.DataFrame()
    last_known_idx = int(np.flatnonzero(known_mask.to_numpy())[-1])
    known = df.loc[:last_known_idx].copy()
    target = df.loc[last_known_idx + 1:].copy()
    target = target.loc[target["TVT"].notna()].copy()
    if target.empty:
        return pd.DataFrame()
    if len(target) > MAX_TARGET_ROWS_PER_WELL:
        sample_idx = np.linspace(0, len(target) - 1, MAX_TARGET_ROWS_PER_WELL).round().astype(int)
        target = target.iloc[np.unique(sample_idx)].copy()

    typewell = load_typewell(csv_path.parent, meta_row["wellname"])
    last = known.iloc[-1]
    first = df.iloc[0]
    known_gr = known["GR"].astype(float).clip(0, 300)
    known_gr_finite = known_gr.dropna()
    gr_fill = float(known_gr.tail(300).median()) if known_gr.notna().any() else 0.0
    target_gr = target["GR"].astype(float).clip(0, 300).fillna(gr_fill)

    known_md_span_state = float(known["MD"].iloc[-1] - known["MD"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_delta_state = float(known["TVT_input"].iloc[-1] - known["TVT_input"].iloc[0]) if len(known) > 1 else 0.0
    known_tvt_slope_state = known_tvt_delta_state / max(abs(known_md_span_state), 1.0)
    known_xy_span_state = float(np.hypot(known["X"].iloc[-1] - known["X"].iloc[0], known["Y"].iloc[-1] - known["Y"].iloc[0])) if len(known) > 1 else 0.0
    known_z_drop_state = float(known["Z"].iloc[-1] - known["Z"].iloc[0]) if len(known) > 1 else 0.0
    known_gr_coverage_state = float(known_gr.notna().mean()) if len(known_gr) else 0.0
    known_gr_last_state = float(known_gr_finite.iloc[-1]) if len(known_gr_finite) else gr_fill
    known_gr_iqr_state = float(known_gr_finite.quantile(0.75) - known_gr_finite.quantile(0.25)) if len(known_gr_finite) >= 4 else 0.0

    target = target.reset_index(drop=False).rename(columns={"index": "row_idx_original"})
    out = pd.DataFrame({
        "wellname": meta_row["wellname"],
        "row_idx_original": target["row_idx_original"].astype(float),
        "MD": target["MD"].astype(float),
        "X": target["X"].astype(float),
        "Y": target["Y"].astype(float),
        "Z": target["Z"].astype(float),
        "GR": target_gr.to_numpy(),
        "TVT": target["TVT"].astype(float),
        "last_known_tvt_feature": float(last["TVT"]),
        "last_known_md": float(last["MD"]),
        "last_known_x": float(last["X"]),
        "last_known_y": float(last["Y"]),
        "last_known_z": float(last["Z"]),
        "first_md": float(first["MD"]),
        "state_known_tvt_last": float(last["TVT_input"] if pd.notna(last.get("TVT_input", np.nan)) else last["TVT"]),
        "state_known_tvt_slope": known_tvt_slope_state,
        "state_known_gr_coverage": known_gr_coverage_state,
        "state_known_gr_last": known_gr_last_state,
        "state_known_gr_iqr": known_gr_iqr_state,
        "state_known_xy_span": known_xy_span_state,
        "state_known_z_drop": known_z_drop_state,
        "state_known_md_span": known_md_span_state,
    })
    out["target_delta"] = out["TVT"] - out["last_known_tvt_feature"]
    out["md_from_last_known"] = out["MD"] - out["last_known_md"]
    out["md_from_start"] = out["MD"] - out["first_md"]
    out["eval_progress"] = safe_divide(np.arange(1, len(out) + 1), len(out))
    out["eval_progress_sq"] = out["eval_progress"] ** 2
    out["dx_from_last"] = out["X"] - out["last_known_x"]
    out["dy_from_last"] = out["Y"] - out["last_known_y"]
    out["dz_from_last"] = out["Z"] - out["last_known_z"]
    out["horizontal_from_last"] = np.sqrt(out["dx_from_last"] ** 2 + out["dy_from_last"] ** 2)
    out["distance_from_last"] = np.sqrt(out["horizontal_from_last"] ** 2 + out["dz_from_last"] ** 2)
    out["distance_log1p"] = np.log1p(out["distance_from_last"].clip(lower=0))
    out["tvt_linear_proxy"] = out["last_known_tvt_feature"] + out["md_from_last_known"]
    best_lag = float(meta_row.get("typewell_best_lag_tvt", 0.0) if pd.notna(meta_row.get("typewell_best_lag_tvt", np.nan)) else 0.0)
    out["typewell_gr_linear_tvt"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy())
    out["typewell_gr_lag_best"] = interp_typewell_gr(typewell, out["tvt_linear_proxy"].to_numpy() + best_lag)
    out["gr_minus_typewell_linear"] = out["GR"] - out["typewell_gr_linear_tvt"]
    out["gr_minus_typewell_bestlag"] = out["GR"] - out["typewell_gr_lag_best"]

    full_gr = df["GR"].astype(float).clip(0, 300).fillna(gr_fill).to_numpy()
    local_rows = []
    for row_idx, linear_tvt in zip(out["row_idx_original"].astype(int).to_numpy(), out["tvt_linear_proxy"].to_numpy(float)):
        target_window = local_gr_window(full_gr, row_idx, radius=16)
        typewell_linear = typewell_window_gr(typewell, linear_tvt, radius=40.0, length=33)
        typewell_lag = typewell_window_gr(typewell, linear_tvt + best_lag, radius=40.0, length=33)
        raw_row = {
            "local_gr_mean_33": float(np.nanmean(target_window)) if np.isfinite(target_window).any() else 0.0,
            "local_gr_std_33": float(np.nanstd(target_window)) if np.isfinite(target_window).any() else 0.0,
            "local_gr_slope_33": local_slope(target_window),
            "local_typewell_dtw_linear": compact_dtw_distance(target_window, typewell_linear),
            "local_typewell_dtw_bestlag": compact_dtw_distance(target_window, typewell_lag),
            "local_typewell_corr_linear": safe_window_corr(target_window, typewell_linear),
            "local_typewell_corr_bestlag": safe_window_corr(target_window, typewell_lag),
        }
        bin_idx = np.linspace(0, len(target_window) - 1, 17).round().astype(int)
        target_center = np.nanmean(target_window) if np.isfinite(target_window).any() else 0.0
        target_scale = np.nanstd(target_window) if np.nanstd(target_window) > 1e-6 else 1.0
        typewell_center = np.nanmean(typewell_lag) if np.isfinite(typewell_lag).any() else 0.0
        for pos, raw_idx in enumerate(bin_idx):
            target_val = float(target_window[raw_idx]) if np.isfinite(target_window[raw_idx]) else target_center
            typewell_val = float(typewell_lag[raw_idx]) if np.isfinite(typewell_lag[raw_idx]) else typewell_center
            raw_row[f"raw_gr_bin_{pos:02d}"] = (target_val - target_center) / target_scale
            raw_row[f"raw_typewell_bin_{pos:02d}"] = typewell_val / 150.0
            raw_row[f"raw_gr_minus_typewell_bin_{pos:02d}"] = (target_val - typewell_val) / 150.0
        local_rows.append(raw_row)
    local_features = pd.DataFrame(local_rows, index=out.index)
    out = pd.concat([out, local_features], axis=1)
    out["local_typewell_dtw_min"] = out[["local_typewell_dtw_linear", "local_typewell_dtw_bestlag"]].min(axis=1)
    out["local_typewell_corr_max"] = out[["local_typewell_corr_linear", "local_typewell_corr_bestlag"]].max(axis=1)
    out["local_typewell_alignment_margin"] = out["local_typewell_corr_bestlag"] - out["local_typewell_corr_linear"]

    for formation in FORMATION_COLS:
        out[f"formation_top_{formation}"] = target[formation].astype(float).to_numpy()
        out[f"label_signed_offset_{formation}"] = out["Z"] - out[f"formation_top_{formation}"]
        out[f"label_abs_offset_{formation}"] = np.abs(out[f"label_signed_offset_{formation}"])

    abs_label_cols = [f"label_abs_offset_{formation}" for formation in FORMATION_COLS]
    out["label_nearest_abs_offset"] = out[abs_label_cols].min(axis=1)
    out["label_nearest_formation"] = out[abs_label_cols].idxmin(axis=1).str.replace("label_abs_offset_", "", regex=False)
    out["label_mean_abs_offset"] = out[abs_label_cols].mean(axis=1)

    for col in [PRIMARY_FOLD, STRESS_FOLD, "descriptor_cluster_fold", "extrapolation_fold"]:
        out[col] = int(meta_row[col])
    for col in DESCRIPTOR_COLS:
        out[col] = float(meta_row[col]) if pd.notna(meta_row[col]) else np.nan
    return out


frames = []
for meta_row in well_meta.to_dict(orient="records"):
    csv_path = TRAIN_DIR / f"{meta_row['wellname']}__horizontal_well.csv"
    if csv_path.exists():
        frame = build_target_rows(csv_path, meta_row)
        if not frame.empty:
            frames.append(frame)

data = pd.concat(frames, ignore_index=True)


def add_selective_calibration(frame):
    out = frame.sort_values(["wellname", "eval_progress"]).copy()
    grouped = out.groupby("wellname", sort=False)
    context_cols = [
        "GR", "gr_minus_typewell_bestlag", "local_typewell_dtw_min", "local_typewell_corr_max",
        "distance_log1p", "md_from_last_known", "horizontal_from_last", "dz_from_last",
    ]
    context_cols = [col for col in context_cols if col in out.columns]
    new_cols = []
    for col in context_cols:
        values = grouped[col]
        out[f"traj_{col}_mean"] = values.transform("mean")
        out[f"traj_{col}_std"] = values.transform("std").fillna(0.0)
        out[f"traj_{col}_centered"] = out[col] - out[f"traj_{col}_mean"]
        out[f"traj_{col}_rank_pct"] = values.rank(pct=True).to_numpy(float)
        out[f"traj_{col}_lag1"] = values.shift(1).fillna(out[col]).to_numpy(float)
        out[f"traj_{col}_lead1"] = values.shift(-1).fillna(out[col]).to_numpy(float)
        out[f"traj_{col}_diff_prev"] = out[col] - out[f"traj_{col}_lag1"]
        out[f"traj_{col}_diff_next"] = out[f"traj_{col}_lead1"] - out[col]
        out[f"traj_{col}_roll5_mean"] = values.transform(lambda s: s.rolling(5, min_periods=1, center=True).mean())
        out[f"traj_{col}_roll5_std"] = values.transform(lambda s: s.rolling(5, min_periods=2, center=True).std()).fillna(0.0)
        out[f"traj_{col}_roll9_mean"] = values.transform(lambda s: s.rolling(9, min_periods=1, center=True).mean())
        out[f"traj_{col}_roll9_centered"] = out[col] - out[f"traj_{col}_roll9_mean"]
        new_cols.extend([
            f"traj_{col}_mean", f"traj_{col}_std", f"traj_{col}_centered", f"traj_{col}_rank_pct",
            f"traj_{col}_diff_prev", f"traj_{col}_diff_next", f"traj_{col}_roll5_mean", f"traj_{col}_roll5_std",
            f"traj_{col}_roll9_mean", f"traj_{col}_roll9_centered",
        ])

    out["traj_progress_rank"] = grouped["eval_progress"].rank(pct=True).to_numpy(float)
    out["traj_row_count"] = grouped["eval_progress"].transform("size").astype(float)
    out["traj_progress_from_mid"] = out["eval_progress"] - 0.5
    out["traj_is_early"] = (out["eval_progress"] <= 0.25).astype(float)
    out["traj_is_late"] = (out["eval_progress"] >= 0.75).astype(float)

    trend_cols = [c for c in ["GR", "gr_minus_typewell_bestlag", "local_typewell_dtw_min", "distance_log1p"] if c in out.columns]
    for col in trend_cols:
        early_values = out.loc[out["eval_progress"] <= 0.33].groupby("wellname")[col].mean()
        late_values = out.loc[out["eval_progress"] >= 0.67].groupby("wellname")[col].mean()
        well_mean = grouped[col].mean()
        out[f"traj_{col}_early_mean"] = out["wellname"].map(early_values).fillna(out["wellname"].map(well_mean)).astype(float)
        out[f"traj_{col}_late_mean"] = out["wellname"].map(late_values).fillna(out["wellname"].map(well_mean)).astype(float)
        out[f"traj_{col}_late_minus_early"] = out[f"traj_{col}_late_mean"] - out[f"traj_{col}_early_mean"]
        out[f"traj_progress_x_{col}_trend"] = out["eval_progress"] * out[f"traj_{col}_late_minus_early"]
        new_cols.extend([
            f"traj_{col}_early_mean", f"traj_{col}_late_mean", f"traj_{col}_late_minus_early", f"traj_progress_x_{col}_trend",
        ])

    out["traj_progress_x_gr_centered"] = out["eval_progress"] * out.get("traj_GR_centered", 0.0)
    out["traj_progress_x_typewell_mismatch"] = out["eval_progress"] * out.get("traj_gr_minus_typewell_bestlag_centered", 0.0)
    out["traj_late_x_typewell_mismatch"] = out["traj_is_late"] * out.get("traj_gr_minus_typewell_bestlag_centered", 0.0)
    out["traj_mid_x_dtw"] = (1.0 - 2.0 * np.abs(out["eval_progress"] - 0.5)).clip(lower=0.0) * out.get("traj_local_typewell_dtw_min_centered", 0.0)
    new_cols.extend([
        "traj_progress_rank", "traj_row_count", "traj_progress_from_mid", "traj_is_early", "traj_is_late",
        "traj_progress_x_gr_centered", "traj_progress_x_typewell_mismatch", "traj_late_x_typewell_mismatch", "traj_mid_x_dtw",
    ])
    out = out.sort_index()
    return out, [col for col in new_cols if col in out.columns]


data, TRAJECTORY_FEATURE_COLS = add_selective_calibration(data)

SAFE_ROW_COLS = [
    "row_idx_original", "MD", "X", "Y", "Z", "GR", "last_known_tvt_feature", "last_known_md",
    "md_from_last_known", "md_from_start", "eval_progress", "eval_progress_sq",
    "dx_from_last", "dy_from_last", "dz_from_last", "horizontal_from_last", "distance_from_last", "distance_log1p",
]
SAFE_FEATURE_COLS = [col for col in [*SAFE_ROW_COLS, *DESCRIPTOR_COLS] if col in data.columns]
TYPEWELL_FEATURE_COLS = [
    "tvt_linear_proxy", "typewell_gr_linear_tvt", "typewell_gr_lag_best", "gr_minus_typewell_linear", "gr_minus_typewell_bestlag",
]
TYPEWELL_FEATURE_COLS = [col for col in TYPEWELL_FEATURE_COLS if col in data.columns]
ALIGNMENT_FEATURE_COLS = [
    "local_gr_mean_33", "local_gr_std_33", "local_gr_slope_33",
    "local_typewell_dtw_linear", "local_typewell_dtw_bestlag", "local_typewell_dtw_min",
    "local_typewell_corr_linear", "local_typewell_corr_bestlag", "local_typewell_corr_max",
    "local_typewell_alignment_margin",
]
ALIGNMENT_FEATURE_COLS = [col for col in ALIGNMENT_FEATURE_COLS if col in data.columns]
RAW_WINDOW_FEATURE_COLS = [
    *[f"raw_gr_bin_{idx:02d}" for idx in range(17)],
    *[f"raw_typewell_bin_{idx:02d}" for idx in range(17)],
    *[f"raw_gr_minus_typewell_bin_{idx:02d}" for idx in range(17)],
]
RAW_WINDOW_FEATURE_COLS = [col for col in RAW_WINDOW_FEATURE_COLS if col in data.columns]
STATE_FEATURE_COLS = [
    "state_known_tvt_last", "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last",
    "state_known_gr_iqr", "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]
DIRECT_FORMATION_FEATURE_COLS = [
    *[f"label_signed_offset_{formation}" for formation in FORMATION_COLS],
    *[f"label_abs_offset_{formation}" for formation in FORMATION_COLS],
    "label_nearest_abs_offset", "label_mean_abs_offset",
]

top_label_summary = []
for formation in FORMATION_COLS:
    values = data[f"formation_top_{formation}"].astype(float)
    offsets = data[f"label_signed_offset_{formation}"].astype(float)
    top_label_summary.append({
        "formation": formation,
        "top_mean": float(values.mean()),
        "top_std": float(values.std()),
        "top_min": float(values.min()),
        "top_max": float(values.max()),
        "signed_offset_mean": float(offsets.mean()),
        "signed_offset_std": float(offsets.std()),
        "abs_offset_mean": float(np.abs(offsets).mean()),
    })
top_label_summary = pd.DataFrame(top_label_summary)
label_counts = data["label_nearest_formation"].value_counts().rename_axis("formation").reset_index(name="nearest_label_rows")
top_label_summary = top_label_summary.merge(label_counts, on="formation", how="left").fillna({"nearest_label_rows": 0})

label_path = TABLE_DIR / "model__cnn_regime_selective_calibration_top_label_summary__2026-07-01.csv"
top_label_summary.to_csv(label_path, index=False)
shutil.copy2(label_path, RUN_DIR / label_path.name)

feature_preview_path = TABLE_DIR / "model__cnn_regime_selective_calibration_feature_preview__2026-07-01.csv"
data.head(500).to_csv(feature_preview_path, index=False)
shutil.copy2(feature_preview_path, RUN_DIR / feature_preview_path.name)

print(data.shape)
print("safe_features", len(SAFE_FEATURE_COLS))
print("alignment_features", len(ALIGNMENT_FEATURE_COLS))
print("raw_window_features", len(RAW_WINDOW_FEATURE_COLS))
print("trajectory_features", len(TRAJECTORY_FEATURE_COLS))
print(top_label_summary.to_string(index=False))

C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:268: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["traj_progress_from_mid"] = out["eval_progress"] - 0.5
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:269: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out["traj_is_early"] = (out["eval_progress"] <= 0.25).astype(float)
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:270: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many 

C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_mean"] = out["wellname"].map(late_values).fillna(out["wellname"].map(well_mean)).astype(float)
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_minus_early"] = out[f"traj_{col}_late_mean"] - out[f"traj_{col}_early_mean"]
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:280: PerformanceWarning: Da

C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_mean"] = out["wellname"].map(late_values).fillna(out["wellname"].map(well_mean)).astype(float)
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_minus_early"] = out[f"traj_{col}_late_mean"] - out[f"traj_{col}_early_mean"]
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:280: PerformanceWarning: Da

C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_mean"] = out["wellname"].map(late_values).fillna(out["wellname"].map(well_mean)).astype(float)
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_minus_early"] = out[f"traj_{col}_late_mean"] - out[f"traj_{col}_early_mean"]
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:280: PerformanceWarning: Da

C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:278: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_mean"] = out["wellname"].map(late_values).fillna(out["wellname"].map(well_mean)).astype(float)
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:279: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  out[f"traj_{col}_late_minus_early"] = out[f"traj_{col}_late_mean"] - out[f"traj_{col}_early_mean"]
C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\2296408312.py:280: PerformanceWarning: Da

(92760, 263)
safe_features 36
alignment_features 10
raw_window_features 51
trajectory_features 105
formation     top_mean    top_std   top_min  top_max  signed_offset_mean  signed_offset_std  abs_offset_mean  nearest_label_rows
     ANCC -9008.099049 633.914859 -10647.31 -7301.76         -416.172460          55.411650       416.172460                 0.0
    ASTNU -9190.780447 639.418693 -10852.79 -7509.20         -244.653750          63.560087       244.653750                 0.0
    ASTNL -9255.299324 624.857096 -10865.58 -7614.43         -180.134874          51.475069       180.135674              2455.0
    EGFDU -9349.765034 619.827567 -10887.58 -7704.03          -85.669164          43.889718        91.053770              3919.0
    EGFDL -9391.937722 619.776843 -10932.35 -7731.61          -43.798726          44.572447        55.375403             50858.0
     BUDA -9516.381964 633.862561 -11104.73 -7835.57           80.947766          45.807212        80.953400             35528.

## 4. Pseudo-Top Probabilities

In [3]:
"Train fold-safe top-elevation ensembles and derive uncertainty, margin, and nearest-top features."

PSEUDO_TOP_PREDICTOR_COLS = [
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *ALIGNMENT_FEATURE_COLS,
    *STATE_FEATURE_COLS,
    *TRAJECTORY_FEATURE_COLS,
]
PSEUDO_TOP_PREDICTOR_COLS = list(dict.fromkeys([col for col in PSEUDO_TOP_PREDICTOR_COLS if col in data.columns]))

pseudo_params = {
    "objective": "reg:squarederror",
    "n_estimators": 220,
    "learning_rate": 0.040,
    "max_depth": 3,
    "min_child_weight": 90,
    "subsample": 0.85,
    "colsample_bytree": 0.85,
    "reg_alpha": 0.20,
    "reg_lambda": 10.0,
    "tree_method": "hist",
    "n_jobs": 4,
    "verbosity": 0,
}


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def softmax_neg_abs(abs_matrix, scale=50.0):
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    exp_logits = np.exp(logits)
    return exp_logits / np.clip(exp_logits.sum(axis=1, keepdims=True), 1e-12, None)


def add_pseudo_features_for_fold_column(frame, fold_col):
    out = frame.copy()
    proxy_quality_rows = []
    for formation in FORMATION_COLS:
        out[f"pseudo_top_elev_{formation}__{fold_col}"] = np.nan
        out[f"pseudo_top_std_{formation}__{fold_col}"] = np.nan

    X_all = out[PSEUDO_TOP_PREDICTOR_COLS].replace([np.inf, -np.inf], np.nan)
    for fold in sorted(out[fold_col].dropna().unique()):
        valid_mask = out[fold_col].eq(fold)
        train_mask = ~valid_mask
        X_train = X_all.loc[train_mask]
        X_valid = X_all.loc[valid_mask]
        for formation in FORMATION_COLS:
            target_col = f"formation_top_{formation}"
            label_train_mask = train_mask & out[target_col].notna()
            label_valid_mask = valid_mask & out[target_col].notna()
            seed_predictions = []
            if label_train_mask.sum() >= 20:
                X_label_train = X_all.loc[label_train_mask]
                y_train = out.loc[label_train_mask, target_col].astype(float)
                for seed in ENSEMBLE_SEEDS:
                    model = XGBRegressor(**pseudo_params, random_state=seed)
                    model.fit(X_label_train, y_train, verbose=False)
                    seed_predictions.append(model.predict(X_valid))
                pred_matrix = np.column_stack(seed_predictions)
                pred_mean = pred_matrix.mean(axis=1)
                pred_std = pred_matrix.std(axis=1)
            else:
                pred_mean = np.full(valid_mask.sum(), float(out[target_col].median()))
                pred_std = np.zeros(valid_mask.sum())
            out.loc[valid_mask, f"pseudo_top_elev_{formation}__{fold_col}"] = pred_mean
            out.loc[valid_mask, f"pseudo_top_std_{formation}__{fold_col}"] = pred_std
            if label_valid_mask.sum() > 0:
                valid_index = out.loc[valid_mask].index
                pred_series = pd.Series(pred_mean, index=valid_index).loc[label_valid_mask]
                y_valid = out.loc[label_valid_mask, target_col].astype(float)
                true_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - y_valid.to_numpy(float)
                pred_offset = out.loc[label_valid_mask, "Z"].to_numpy(float) - pred_series.to_numpy(float)
                corr = float(np.corrcoef(y_valid.to_numpy(float), pred_series.to_numpy(float))[0, 1]) if len(y_valid) > 2 else np.nan
                top_rmse = rmse(y_valid, pred_series)
                top_mae = float(mean_absolute_error(y_valid, pred_series))
                signed_offset_rmse = rmse(true_offset, pred_offset)
            else:
                corr = np.nan
                top_rmse = np.nan
                top_mae = np.nan
                signed_offset_rmse = np.nan
            proxy_quality_rows.append({
                "fold_column": fold_col,
                "fold": int(fold),
                "formation": formation,
                "n_train": int(label_train_mask.sum()),
                "n_valid": int(label_valid_mask.sum()),
                "top_rmse": top_rmse,
                "top_mae": top_mae,
                "top_corr": corr,
                "signed_offset_rmse": signed_offset_rmse,
                "mean_ensemble_std": float(np.mean(pred_std)),
            })

    for formation in FORMATION_COLS:
        top_col = f"pseudo_top_elev_{formation}__{fold_col}"
        std_col = f"pseudo_top_std_{formation}__{fold_col}"
        out[top_col] = out[top_col].fillna(float(out[f"formation_top_{formation}"].median()))
        out[std_col] = out[std_col].fillna(float(out[std_col].median()) if out[std_col].notna().any() else 0.0)
        out[f"pseudo_signed_offset_{formation}__{fold_col}"] = out["Z"] - out[top_col]
        out[f"pseudo_abs_offset_{formation}__{fold_col}"] = np.abs(out[f"pseudo_signed_offset_{formation}__{fold_col}"])

    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    std_cols = [f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    probs = softmax_neg_abs(abs_matrix)
    sorted_abs = np.sort(abs_matrix, axis=1)
    sorted_probs = np.sort(probs, axis=1)[:, ::-1]
    nearest_idx = np.argmin(abs_matrix, axis=1)

    out[f"pseudo_nearest_abs_offset__{fold_col}"] = sorted_abs[:, 0]
    out[f"pseudo_second_abs_offset__{fold_col}"] = sorted_abs[:, 1]
    out[f"pseudo_top_margin__{fold_col}"] = out[f"pseudo_second_abs_offset__{fold_col}"] - out[f"pseudo_nearest_abs_offset__{fold_col}"]
    out[f"pseudo_top_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp(-out[f"pseudo_top_margin__{fold_col}"].clip(-100, 100) / 25.0))
    out[f"pseudo_near_boundary_25__{fold_col}"] = (out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 25.0).astype(float)
    out[f"pseudo_uncertain_boundary__{fold_col}"] = ((out[f"pseudo_nearest_abs_offset__{fold_col}"] <= 50.0) & (out[f"pseudo_top_margin__{fold_col}"] <= 25.0)).astype(float)
    out[f"pseudo_mean_abs_offset__{fold_col}"] = abs_matrix.mean(axis=1)
    out[f"pseudo_min_std__{fold_col}"] = out[std_cols].min(axis=1)
    out[f"pseudo_mean_std__{fold_col}"] = out[std_cols].mean(axis=1)
    out[f"pseudo_top1_prob__{fold_col}"] = sorted_probs[:, 0]
    out[f"pseudo_top2_prob__{fold_col}"] = sorted_probs[:, 1]
    out[f"pseudo_top_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"pseudo_nearest_formation__{fold_col}"] = [FORMATION_COLS[int(idx)] for idx in nearest_idx]
    out[f"pseudo_nearest_matches_label__{fold_col}"] = (out[f"pseudo_nearest_formation__{fold_col}"] == out["label_nearest_formation"]).astype(float)

    return out, pd.DataFrame(proxy_quality_rows)


pseudo_feature_cols_by_fold = {}
pseudo_uncertainty_cols_by_fold = {}
quality_parts = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    data, quality = add_pseudo_features_for_fold_column(data, fold_col)
    quality_parts.append(quality)
    pseudo_feature_cols_by_fold[fold_col] = [
        *[f"pseudo_signed_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        *[f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_nearest_abs_offset__{fold_col}",
        f"pseudo_second_abs_offset__{fold_col}",
        f"pseudo_mean_abs_offset__{fold_col}",
    ]
    pseudo_uncertainty_cols_by_fold[fold_col] = [
        *[f"pseudo_top_std_{formation}__{fold_col}" for formation in FORMATION_COLS],
        f"pseudo_top_margin__{fold_col}",
        f"pseudo_top_confidence__{fold_col}",
        f"pseudo_near_boundary_25__{fold_col}",
        f"pseudo_uncertain_boundary__{fold_col}",
        f"pseudo_min_std__{fold_col}",
        f"pseudo_mean_std__{fold_col}",
        f"pseudo_top1_prob__{fold_col}",
        f"pseudo_top2_prob__{fold_col}",
        f"pseudo_top_entropy__{fold_col}",
    ]

proxy_quality = pd.concat(quality_parts, ignore_index=True)
nearest_quality = []
for fold_col in [PRIMARY_FOLD, STRESS_FOLD]:
    nearest_quality.append({
        "fold_column": fold_col,
        "fold": "all",
        "formation": "nearest_classifier",
        "n_train": np.nan,
        "n_valid": int(len(data)),
        "top_rmse": np.nan,
        "top_mae": np.nan,
        "top_corr": np.nan,
        "signed_offset_rmse": np.nan,
        "mean_ensemble_std": float(data[f"pseudo_mean_std__{fold_col}"].mean()),
        "nearest_accuracy": float(data[f"pseudo_nearest_matches_label__{fold_col}"].mean()),
    })
proxy_quality["nearest_accuracy"] = np.nan
proxy_quality = pd.concat([proxy_quality, pd.DataFrame(nearest_quality)], ignore_index=True)

proxy_quality_path = TABLE_DIR / "model__cnn_regime_selective_calibration_proxy_quality__2026-07-01.csv"
proxy_quality.to_csv(proxy_quality_path, index=False)
shutil.copy2(proxy_quality_path, RUN_DIR / proxy_quality_path.name)
proxy_quality.groupby(["fold_column", "formation"], dropna=False).agg(top_rmse=("top_rmse", "mean"), top_mae=("top_mae", "mean"), mean_ensemble_std=("mean_ensemble_std", "mean"), nearest_accuracy=("nearest_accuracy", "mean")).reset_index()

,fold_column,formation,top_rmse,top_mae,mean_ensemble_std,nearest_accuracy
0,analog_distance_fold,ANCC,36.469822,27.765136,3.181215,NaN
1,analog_distance_fold,ASTNL,36.989040,27.762242,3.552037,NaN
2,analog_distance_fold,ASTNU,38.521277,28.799564,3.483139,NaN
3,analog_distance_fold,BUDA,34.429892,25.357459,3.157898,NaN
4,analog_distance_fold,EGFDL,34.673484,25.931932,3.234797,NaN
5,analog_distance_fold,EGFDU,34.336530,25.539977,3.111420,NaN
6,analog_distance_fold,nearest_classifier,NaN,NaN,3.286660,0.575841
7,strict_hybrid_fold,ANCC,35.005949,26.872908,3.176805,NaN
8,strict_hybrid_fold,ASTNL,35.819052,27.471610,3.687508,NaN
9,strict_hybrid_fold,ASTNU,37.521710,28.493977,3.626344,NaN


## 5. Regime Probability Basis

In [4]:
"Create soft formation-regime probabilities and MAP residual basis features."


def add_probability_and_basis_features(frame, fold_col, scale=50.0):
    out = frame
    abs_cols = [f"pseudo_abs_offset_{formation}__{fold_col}" for formation in FORMATION_COLS]
    abs_matrix = out[abs_cols].to_numpy(float)
    logits = -abs_matrix / scale
    logits = logits - np.nanmax(logits, axis=1, keepdims=True)
    probs = np.exp(logits)
    probs = probs / np.clip(probs.sum(axis=1, keepdims=True), 1e-12, None)
    for idx, formation in enumerate(FORMATION_COLS):
        prob_col = f"mix_prob_{formation}__{fold_col}"
        out[prob_col] = probs[:, idx]
        out[f"mix_prob_{formation}_x_progress__{fold_col}"] = out[prob_col] * out["eval_progress"]
        out[f"mix_prob_{formation}_x_distance__{fold_col}"] = out[prob_col] * out["distance_log1p"]
        out[f"mix_prob_{formation}_x_near_top__{fold_col}"] = out[prob_col] * (out[f"pseudo_abs_offset_{formation}__{fold_col}"] <= 40.0).astype(float)
    out[f"mix_entropy__{fold_col}"] = -(probs * np.log(probs + 1e-12)).sum(axis=1)
    out[f"mix_top1_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -1]
    out[f"mix_top2_prob__{fold_col}"] = np.sort(probs, axis=1)[:, -2]
    out[f"mix_prob_margin__{fold_col}"] = out[f"mix_top1_prob__{fold_col}"] - out[f"mix_top2_prob__{fold_col}"]
    out[f"mix_confidence__{fold_col}"] = 1.0 / (1.0 + np.exp((out[f"mix_entropy__{fold_col}"] - 1.15) / 0.12))
    return out


mix_basis_cols_by_fold = {}
mix_uncertainty_cols_by_fold = {}
for active_fold in [PRIMARY_FOLD, STRESS_FOLD]:
    data = add_probability_and_basis_features(data, active_fold)
    mix_basis_cols_by_fold[active_fold] = []
    for formation in FORMATION_COLS:
        mix_basis_cols_by_fold[active_fold].extend([
            f"mix_prob_{formation}__{active_fold}",
            f"mix_prob_{formation}_x_progress__{active_fold}",
            f"mix_prob_{formation}_x_distance__{active_fold}",
            f"mix_prob_{formation}_x_near_top__{active_fold}",
        ])
    mix_uncertainty_cols_by_fold[active_fold] = [
        f"mix_entropy__{active_fold}",
        f"mix_top1_prob__{active_fold}",
        f"mix_top2_prob__{active_fold}",
        f"mix_prob_margin__{active_fold}",
        f"mix_confidence__{active_fold}",
        f"pseudo_top_margin__{active_fold}",
        f"pseudo_mean_std__{active_fold}",
        f"pseudo_top_entropy__{active_fold}",
    ]

print(len(mix_basis_cols_by_fold[PRIMARY_FOLD]), len(mix_uncertainty_cols_by_fold[PRIMARY_FOLD]))
data[mix_basis_cols_by_fold[PRIMARY_FOLD] + mix_uncertainty_cols_by_fold[PRIMARY_FOLD]].head()

24 8


,mix_prob_ANCC__strict_hybrid_fold,mix_prob_ANCC_x_progress__strict_hybrid_fold,mix_prob_ANCC_x_distance__strict_hybrid_fold,mix_prob_ANCC_x_near_top__strict_hybrid_fold,mix_prob_ASTNU__strict_hybrid_fold,mix_prob_ASTNU_x_progress__strict_hybrid_fold,mix_prob_ASTNU_x_distance__strict_hybrid_fold,mix_prob_ASTNU_x_near_top__strict_hybrid_fold,mix_prob_ASTNL__strict_hybrid_fold,mix_prob_ASTNL_x_progress__strict_hybrid_fold,mix_prob_ASTNL_x_distance__strict_hybrid_fold,mix_prob_ASTNL_x_near_top__strict_hybrid_fold,mix_prob_EGFDU__strict_hybrid_fold,mix_prob_EGFDU_x_progress__strict_hybrid_fold,mix_prob_EGFDU_x_distance__strict_hybrid_fold,mix_prob_EGFDU_x_near_top__strict_hybrid_fold,mix_prob_EGFDL__strict_hybrid_fold,mix_prob_EGFDL_x_progress__strict_hybrid_fold,mix_prob_EGFDL_x_distance__strict_hybrid_fold,mix_prob_EGFDL_x_near_top__strict_hybrid_fold,mix_prob_BUDA__strict_hybrid_fold,mix_prob_BUDA_x_progress__strict_hybrid_fold,mix_prob_BUDA_x_distance__strict_hybrid_fold,mix_prob_BUDA_x_near_top__strict_hybrid_fold,mix_entropy__strict_hybrid_fold,mix_top1_prob__strict_hybrid_fold,mix_top2_prob__strict_hybrid_fold,mix_prob_margin__strict_hybrid_fold,mix_confidence__strict_hybrid_fold,pseudo_top_margin__strict_hybrid_fold,pseudo_mean_std__strict_hybrid_fold,pseudo_top_entropy__strict_hybrid_fold
0,0.000418,0.000003,0.000290,0.0,0.014928,0.000124,0.010351,0.0,0.016882,0.000141,0.011706,0.0,0.100728,0.000839,0.069845,0.0,0.223126,0.001859,0.154715,0.0,0.643917,0.005366,0.446490,0.643917,0.984265,0.643917,0.223126,0.420790,0.799171,52.991562,4.741536,0.984265
1,0.000424,0.000007,0.001494,0.0,0.015113,0.000252,0.053294,0.0,0.017091,0.000285,0.060269,0.0,0.101977,0.001700,0.359601,0.0,0.225892,0.003765,0.796564,0.0,0.639502,0.010658,2.255077,0.639502,0.990968,0.639502,0.225892,0.413610,0.790057,52.031563,4.741536,0.990968
2,0.000432,0.000011,0.001811,0.0,0.015420,0.000385,0.064604,0.0,0.017438,0.000436,0.073060,0.0,0.104047,0.002601,0.435919,0.0,0.230477,0.005762,0.965619,0.0,0.632186,0.015805,2.648638,0.632186,1.001891,0.632186,0.230477,0.401709,0.774560,50.451562,4.741536,1.001891
3,0.000448,0.000015,0.002060,0.0,0.016520,0.000551,0.075910,0.0,0.019326,0.000644,0.088803,0.0,0.109185,0.003640,0.501709,0.0,0.239402,0.007980,1.100058,0.0,0.615118,0.020504,2.826481,0.615118,1.030483,0.615118,0.239402,0.375716,0.730266,47.183477,4.745036,1.030483
4,0.000455,0.000019,0.002217,0.0,0.016224,0.000676,0.079093,0.0,0.018347,0.000764,0.089446,0.0,0.109470,0.004561,0.533683,0.0,0.242490,0.010104,1.182180,0.0,0.613015,0.025542,2.988559,0.613015,1.029424,0.613015,0.242490,0.370526,0.732001,46.371563,4.741536,1.029424


## 6. Regime Anchor, Dilated CNN Specialist, Selective Trajectory Calibration, Gates, And Tuned Well Bias

In [5]:
"Recompute regime/CNN predictions and train fold-heldout selective trajectory calibration models."

torch.manual_seed(SEED)
torch.set_num_threads(4)
CNN_SEED_OFFSETS = [0, 101, 211]
CNN_EPOCHS = 6
STATIC_BLEND_WEIGHTS = [0.0, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]

SEQ_GROUPS = [
    [f"raw_gr_bin_{idx:02d}" for idx in range(17)],
    [f"raw_typewell_bin_{idx:02d}" for idx in range(17)],
    [f"raw_gr_minus_typewell_bin_{idx:02d}" for idx in range(17)],
]
TABULAR_FEATURE_COLS = list(dict.fromkeys([
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *ALIGNMENT_FEATURE_COLS,
    *TRAJECTORY_FEATURE_COLS,
    "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
    "combined_stress_score", "selected_analog_distance",
]))
TABULAR_FEATURE_COLS = [col for col in TABULAR_FEATURE_COLS if col in data.columns]
BASE_FEATURE_COLS = list(dict.fromkeys([*SAFE_FEATURE_COLS, *TYPEWELL_FEATURE_COLS, *TRAJECTORY_FEATURE_COLS]))
REGIME_FEATURE_COLS = list(dict.fromkeys([
    *SAFE_FEATURE_COLS,
    *TYPEWELL_FEATURE_COLS,
    *TRAJECTORY_FEATURE_COLS,
    "state_known_tvt_slope", "state_known_gr_coverage", "state_known_gr_last", "state_known_gr_iqr",
    "state_known_xy_span", "state_known_z_drop", "state_known_md_span",
]))
REGIME_FEATURE_COLS = [col for col in REGIME_FEATURE_COLS if col in data.columns]


def rmse(y_true, y_pred):
    return float(mean_squared_error(y_true, y_pred) ** 0.5)


def metric_row(actual, pred, baseline):
    return {
        "rmse": rmse(actual, pred),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "last_known_hold_rmse": rmse(actual, baseline),
    }


class LocalWindowCNN(nn.Module):
    def __init__(self, n_tabular):
        super().__init__()
        self.seq = nn.Sequential(
            nn.Conv1d(3, 24, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(24, 32, kernel_size=3, padding=2, dilation=2),
            nn.GELU(),
            nn.Conv1d(32, 40, kernel_size=3, padding=4, dilation=4),
            nn.GELU(),
            nn.Conv1d(40, 40, kernel_size=1),
            nn.GELU(),
            nn.AdaptiveAvgPool1d(1),
        )
        self.tab = nn.Sequential(nn.Linear(n_tabular, 72), nn.GELU(), nn.Dropout(0.12))
        self.head = nn.Sequential(nn.Linear(112, 72), nn.GELU(), nn.Dropout(0.12), nn.Linear(72, 1))

    def forward(self, seq_x, tab_x):
        seq_h = self.seq(seq_x).squeeze(-1)
        tab_h = self.tab(tab_x)
        return self.head(torch.cat([seq_h, tab_h], dim=1)).squeeze(1)


def _standardize(train_values, valid_values):
    mean = np.nanmean(train_values, axis=0)
    mean = np.where(np.isfinite(mean), mean, 0.0)
    std = np.nanstd(train_values, axis=0)
    std = np.where(np.isfinite(std) & (std >= 1e-6), std, 1.0)
    train_filled = np.where(np.isfinite(train_values), train_values, mean)
    valid_filled = np.where(np.isfinite(valid_values), valid_values, mean)
    train_scaled = (train_filled - mean) / std
    valid_scaled = (valid_filled - mean) / std
    return train_scaled.astype("float32"), valid_scaled.astype("float32")


def make_cnn_arrays(frame, train_mask, valid_mask):
    train_seq_channels = []
    valid_seq_channels = []
    for cols in SEQ_GROUPS:
        train_scaled, valid_scaled = _standardize(frame.loc[train_mask, cols].to_numpy(float), frame.loc[valid_mask, cols].to_numpy(float))
        train_seq_channels.append(train_scaled)
        valid_seq_channels.append(valid_scaled)
    train_seq = np.stack(train_seq_channels, axis=1)
    valid_seq = np.stack(valid_seq_channels, axis=1)
    train_tab, valid_tab = _standardize(frame.loc[train_mask, TABULAR_FEATURE_COLS].to_numpy(float), frame.loc[valid_mask, TABULAR_FEATURE_COLS].to_numpy(float))
    y_train = frame.loc[train_mask, "target_delta"].to_numpy(float)
    y_mean = float(np.mean(y_train))
    y_std = float(np.std(y_train) if np.std(y_train) > 1e-6 else 1.0)
    return train_seq, valid_seq, train_tab, valid_tab, ((y_train - y_mean) / y_std).astype("float32"), y_mean, y_std


def train_cnn(train_seq, train_tab, y_train, seed):
    torch.manual_seed(seed)
    model = LocalWindowCNN(train_tab.shape[1])
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=2.0e-4)
    loss_fn = nn.SmoothL1Loss(beta=0.5)
    loader = DataLoader(TensorDataset(torch.from_numpy(train_seq), torch.from_numpy(train_tab), torch.from_numpy(y_train)), batch_size=512, shuffle=True)
    model.train()
    for epoch in range(CNN_EPOCHS):
        for seq_batch, tab_batch, y_batch in loader:
            optimizer.zero_grad()
            loss = loss_fn(model(seq_batch, tab_batch), y_batch)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()
    return model


def predict_cnn(model, valid_seq, valid_tab, y_mean, y_std):
    model.eval()
    preds = []
    with torch.no_grad():
        loader = DataLoader(TensorDataset(torch.from_numpy(valid_seq), torch.from_numpy(valid_tab)), batch_size=2048, shuffle=False)
        for seq_batch, tab_batch in loader:
            preds.append(model(seq_batch, tab_batch).cpu().numpy())
    return np.concatenate(preds) * y_std + y_mean


def fit_global_xgb(X_train, y_train, X_valid):
    model = XGBRegressor(
        objective="reg:squarederror", n_estimators=520, learning_rate=0.036, max_depth=5,
        min_child_weight=90, subsample=0.85, colsample_bytree=0.85, reg_alpha=0.20,
        reg_lambda=8.0, gamma=0.10, tree_method="hist", random_state=SEED, n_jobs=4, verbosity=0,
    )
    model.fit(X_train, y_train, verbose=False)
    return model.predict(X_valid)


def fit_weighted_ridge(X_train, y_train, sample_weight, alpha=80.0):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    ridge = Ridge(alpha=alpha)
    X_imp = imputer.fit_transform(X_train)
    X_scaled = scaler.fit_transform(X_imp)
    ridge.fit(X_scaled, y_train, sample_weight=sample_weight)
    return {"imputer": imputer, "scaler": scaler, "ridge": ridge}


def predict_weighted_ridge(model, X):
    return model["ridge"].predict(model["scaler"].transform(model["imputer"].transform(X)))


def blend_weights(frame, fold_col):
    confidence = frame[f"mix_confidence__{fold_col}"].fillna(frame[f"mix_confidence__{fold_col}"].median()).clip(0.0, 1.0)
    margin = frame[f"mix_prob_margin__{fold_col}"].fillna(frame[f"mix_prob_margin__{fold_col}"].median()).clip(0.0, 1.0)
    entropy = frame[f"mix_entropy__{fold_col}"].fillna(frame[f"mix_entropy__{fold_col}"].median())
    entropy_score = 1.0 / (1.0 + np.exp((entropy - 1.15) / 0.12))
    raw = 0.50 * np.asarray(confidence) + 0.35 * np.asarray(margin) + 0.15 * np.asarray(entropy_score)
    return np.clip(raw, 0.0, 1.0)


def regime_reference_delta(frame, fold_col, train_mask, valid_mask):
    prob_cols = [f"mix_prob_{formation}__{fold_col}" for formation in FORMATION_COLS]
    X_base = frame[BASE_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    X_regime = frame[REGIME_FEATURE_COLS].replace([np.inf, -np.inf], np.nan)
    y_delta = frame["target_delta"].astype(float)
    global_valid_delta = fit_global_xgb(X_base.loc[train_mask], y_delta.loc[train_mask], X_base.loc[valid_mask])
    train_probs = frame.loc[train_mask, prob_cols].to_numpy(float)
    valid_probs = frame.loc[valid_mask, prob_cols].to_numpy(float)
    regime_valid_preds = []
    for idx, formation in enumerate(FORMATION_COLS):
        weights = np.clip(train_probs[:, idx], 1e-4, None)
        weights = weights / np.mean(weights)
        model = fit_weighted_ridge(X_regime.loc[train_mask], y_delta.loc[train_mask], weights, alpha=80.0)
        regime_valid_preds.append(predict_weighted_ridge(model, X_regime.loc[valid_mask]))
    regime_valid_delta = np.sum(valid_probs * np.column_stack(regime_valid_preds), axis=1)
    weights = blend_weights(frame.loc[valid_mask], fold_col)
    return (1.0 - weights) * global_valid_delta + weights * regime_valid_delta


def base_candidate_predictions(fold_col):
    parts = []
    for fold in sorted(data[fold_col].dropna().unique()):
        valid_mask = data[fold_col].eq(fold)
        train_mask = ~valid_mask
        regime_delta = regime_reference_delta(data, fold_col, train_mask, valid_mask)
        train_seq, valid_seq, train_tab, valid_tab, y_train_scaled, y_mean, y_std = make_cnn_arrays(data, train_mask, valid_mask)
        seed_deltas = []
        for offset in CNN_SEED_OFFSETS:
            cnn_model = train_cnn(train_seq, train_tab, y_train_scaled, SEED + int(fold) + offset)
            seed_deltas.append(predict_cnn(cnn_model, valid_seq, valid_tab, y_mean, y_std))
        cnn_delta = np.mean(np.column_stack(seed_deltas), axis=1)
        tmp = data.loc[valid_mask, [
            "wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score",
            f"mix_confidence__{fold_col}", f"mix_entropy__{fold_col}", f"mix_prob_margin__{fold_col}",
            "GR", "distance_log1p", "md_from_last_known", *TRAJECTORY_FEATURE_COLS,
        ]].copy()
        tmp["fold_column"] = fold_col
        tmp["fold"] = int(fold)
        tmp["actual_tvt"] = data.loc[valid_mask, "TVT"].to_numpy(float)
        tmp["baseline_tvt"] = data.loc[valid_mask, "last_known_tvt_feature"].to_numpy(float)
        tmp["regime_pred_tvt_base"] = tmp["baseline_tvt"].to_numpy(float) + regime_delta
        tmp["cnn_pred_tvt_base"] = tmp["baseline_tvt"].to_numpy(float) + cnn_delta
        tmp["pred_gap"] = tmp["cnn_pred_tvt_base"] - tmp["regime_pred_tvt_base"]
        tmp["pred_gap_abs"] = np.abs(tmp["pred_gap"])
        return_cols = [col for cols in SEQ_GROUPS for col in cols]
        tmp["raw_window_energy"] = data.loc[valid_mask, return_cols].abs().mean(axis=1).to_numpy(float)
        parts.append(tmp)
    out = pd.concat(parts, ignore_index=True)
    out = add_gate_context_features(out, fold_col)
    return out



def add_gate_context_features(frame, fold_col):
    out = frame.copy()
    out["progress_bin"] = pd.cut(out["eval_progress"], bins=[-0.001, 0.15, 0.35, 0.55, 0.75, 1.001], labels=False).astype(float)
    out["pred_gap_signed_abs_ratio"] = out["pred_gap"] / (out["pred_gap_abs"] + 1.0)
    out["pred_gap_x_confidence"] = out["pred_gap_abs"] * out[f"mix_confidence__{fold_col}"]
    out["pred_gap_x_entropy"] = out["pred_gap_abs"] * out[f"mix_entropy__{fold_col}"]
    out["stress_x_gap"] = out["combined_stress_score"] * out["pred_gap_abs"]
    out["progress_x_gap"] = out["eval_progress"] * out["pred_gap_abs"]
    out["distance_x_gap"] = out["distance_log1p"] * out["pred_gap_abs"]
    out["cnn_minus_baseline"] = out["cnn_pred_tvt_base"] - out["baseline_tvt"]
    out["regime_minus_baseline"] = out["regime_pred_tvt_base"] - out["baseline_tvt"]
    out["cnn_regime_midpoint"] = 0.5 * (out["cnn_pred_tvt_base"] + out["regime_pred_tvt_base"])
    out["cnn_regime_midpoint_delta"] = out["cnn_regime_midpoint"] - out["baseline_tvt"]

    grouped = out.sort_values(["wellname", "eval_progress"]).groupby("wellname", sort=False)
    for col in ["pred_gap_abs", "pred_gap", f"mix_confidence__{fold_col}", f"mix_entropy__{fold_col}", "raw_window_energy"]:
        values = grouped[col]
        out[f"well_{col}_mean"] = values.transform("mean")
        out[f"well_{col}_std"] = values.transform("std").fillna(0.0)
        out[f"well_{col}_rank_pct"] = values.rank(pct=True).to_numpy(float)
        out[f"well_{col}_centered"] = out[col] - out[f"well_{col}_mean"]
    out["well_row_count"] = grouped["actual_tvt"].transform("size").astype(float)
    out["well_progress_rank"] = grouped["eval_progress"].rank(pct=True).to_numpy(float)
    return out


def add_candidate_rows(base_predictions, fold_col):
    rows = []
    pred_parts = []
    actual_all = base_predictions["actual_tvt"].to_numpy(float)
    baseline_all = base_predictions["baseline_tvt"].to_numpy(float)
    regime_all = base_predictions["regime_pred_tvt_base"].to_numpy(float)
    cnn_all = base_predictions["cnn_pred_tvt_base"].to_numpy(float)
    gap_all = cnn_all - regime_all
    optimal_weight = np.clip((actual_all - regime_all) / np.where(np.abs(gap_all) < 1e-6, np.nan, gap_all), 0.0, 1.0)
    optimal_weight = np.nan_to_num(optimal_weight, nan=0.0, posinf=1.0, neginf=0.0)
    base_predictions["optimal_weight"] = optimal_weight
    candidate_preds = {
        "oracle_continuous_weight": regime_all + optimal_weight * gap_all,
        "oracle_row_switch": np.where(np.abs(actual_all - cnn_all) < np.abs(actual_all - regime_all), cnn_all, regime_all),
    }
    for weight in STATIC_BLEND_WEIGHTS:
        label = "regime_reference" if weight == 0.0 else f"blend_cnn_{int(weight * 100):02d}"
        candidate_preds[label] = regime_all + weight * gap_all

    gate_cols = [
        "eval_progress", "progress_bin", "well_progress_rank", "selected_analog_distance", "combined_stress_score",
        f"mix_confidence__{fold_col}", f"mix_entropy__{fold_col}", f"mix_prob_margin__{fold_col}",
        "GR", "distance_log1p", "md_from_last_known", "pred_gap", "pred_gap_abs", "raw_window_energy",
        "pred_gap_signed_abs_ratio", "pred_gap_x_confidence", "pred_gap_x_entropy", "stress_x_gap", "progress_x_gap", "distance_x_gap",
        "cnn_minus_baseline", "regime_minus_baseline", "cnn_regime_midpoint_delta", "well_row_count",
        *[col for col in TRAJECTORY_FEATURE_COLS if col in base_predictions.columns],
    ]
    gate_cols.extend([col for col in base_predictions.columns if col.startswith("well_pred_gap") or col.startswith("well_mix_") or col.startswith("well_raw_window") or col.startswith("traj_")])
    gate_cols = list(dict.fromkeys([col for col in gate_cols if col in base_predictions.columns]))
    for fold in sorted(base_predictions["fold"].dropna().unique()):
        train_mask = ~base_predictions["fold"].eq(fold).to_numpy()
        valid_mask = base_predictions["fold"].eq(fold).to_numpy()
        train = base_predictions.loc[train_mask]
        valid = base_predictions.loc[valid_mask]
        X_train = train[gate_cols].replace([np.inf, -np.inf], np.nan)
        X_valid = valid[gate_cols].replace([np.inf, -np.inf], np.nan)
        imputer = SimpleImputer(strategy="median")
        X_train_imp = imputer.fit_transform(X_train)
        X_valid_imp = imputer.transform(X_valid)
        cnn_better = (np.abs(train["actual_tvt"].to_numpy(float) - train["cnn_pred_tvt_base"].to_numpy(float)) < np.abs(train["actual_tvt"].to_numpy(float) - train["regime_pred_tvt_base"].to_numpy(float))).astype(float)
        for depth, eta, label_suffix in [(2, 0.030, "soft"), (3, 0.025, "medium")]:
            weight_model = XGBRegressor(
                objective="reg:squarederror", n_estimators=240, learning_rate=eta, max_depth=depth,
                min_child_weight=110, subsample=0.88, colsample_bytree=0.88, reg_alpha=0.55,
                reg_lambda=18.0, gamma=0.15, tree_method="hist", random_state=SEED + int(fold),
                n_jobs=4, verbosity=0,
            )
            weight_model.fit(X_train_imp, train["optimal_weight"].to_numpy(float), verbose=False)
            predicted_weight = np.clip(weight_model.predict(X_valid_imp), 0.0, 1.0)
            for shrink in [0.90, 1.00, 1.10]:
                candidate_preds.setdefault(f"xgb_weight_{label_suffix}_s{int(shrink * 100):03d}", np.full(len(base_predictions), np.nan))
                candidate_preds[f"xgb_weight_{label_suffix}_s{int(shrink * 100):03d}"][valid_mask] = regime_all[valid_mask] + shrink * predicted_weight * gap_all[valid_mask]

        if len(np.unique(cnn_better)) == 2:
            classifier = LogisticRegression(C=0.25, max_iter=1000, class_weight="balanced", random_state=SEED + int(fold))
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_imp)
            X_valid_scaled = scaler.transform(X_valid_imp)
            classifier.fit(X_train_scaled, cnn_better)
            switch_prob = np.clip(classifier.predict_proba(X_valid_scaled)[:, 1], 0.0, 1.0)
            for floor_weight in [0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.22]:
                calibrated_weight = floor_weight + (1.0 - floor_weight) * switch_prob
                label = f"logit_switch_floor{int(floor_weight * 100):02d}"
                candidate_preds.setdefault(label, np.full(len(base_predictions), np.nan))
                candidate_preds[label][valid_mask] = regime_all[valid_mask] + calibrated_weight * gap_all[valid_mask]

            switch_model = XGBRegressor(
                objective="reg:logistic", n_estimators=220, learning_rate=0.026, max_depth=2,
                min_child_weight=130, subsample=0.90, colsample_bytree=0.90, reg_alpha=0.70,
                reg_lambda=20.0, gamma=0.20, tree_method="hist", random_state=SEED + 404 + int(fold),
                n_jobs=4, verbosity=0,
            )
            switch_model.fit(X_train_imp, cnn_better, verbose=False)
            switch_prob = np.clip(switch_model.predict(X_valid_imp), 0.0, 1.0)
            for power in [0.70, 0.85, 0.90, 1.00, 1.10, 1.20]:
                calibrated_weight = np.clip(switch_prob ** power, 0.0, 1.0)
                label = f"xgb_switch_p{int(power * 100):03d}"
                candidate_preds.setdefault(label, np.full(len(base_predictions), np.nan))
                candidate_preds[label][valid_mask] = regime_all[valid_mask] + calibrated_weight * gap_all[valid_mask]

    bias_feature_cols = [
        "selected_analog_distance", "combined_stress_score", f"mix_confidence__{fold_col}", f"mix_entropy__{fold_col}",
        f"mix_prob_margin__{fold_col}", "GR", "distance_log1p", "md_from_last_known", "pred_gap_abs", "raw_window_energy",
        "cnn_minus_baseline", "regime_minus_baseline", "cnn_regime_midpoint_delta", "well_row_count",
        *[col for col in TRAJECTORY_FEATURE_COLS if col in base_predictions.columns],
    ]
    bias_feature_cols = [col for col in bias_feature_cols if col in base_predictions.columns]
    base_well_features = (
        base_predictions[["wellname", "fold", *bias_feature_cols]]
        .groupby("wellname", sort=False)
        .agg({**{col: "mean" for col in bias_feature_cols}, "fold": "first"})
        .reset_index()
    )
    allowed_bias_sources = {
        "logit_switch_floor10",
        "logit_switch_floor12",
        "logit_switch_floor15",
        "logit_switch_floor18",
        "logit_switch_floor20",
        "xgb_weight_soft_s090",
        "xgb_weight_soft_s100",
        "xgb_weight_soft_s110",
        "xgb_switch_p085",
        "xgb_switch_p090",
        "xgb_switch_p100",
        "blend_cnn_50",
        "blend_cnn_55",
    }
    for source_label, source_pred in list(candidate_preds.items()):
        if str(source_label).startswith("oracle_"):
            continue
        if source_label not in allowed_bias_sources:
            continue
        if not np.isfinite(source_pred).all():
            continue
        pred_frame = base_predictions[["wellname", "fold", "actual_tvt"]].copy()
        pred_frame["pred_tvt"] = source_pred
        pred_frame["residual"] = pred_frame["actual_tvt"] - pred_frame["pred_tvt"]
        pred_frame["pred_delta"] = pred_frame["pred_tvt"] - baseline_all
        pred_well = pred_frame.groupby("wellname", sort=False).agg(
            target_bias=("residual", "mean"),
            pred_mean=("pred_tvt", "mean"),
            pred_std=("pred_tvt", "std"),
            pred_delta_mean=("pred_delta", "mean"),
            pred_delta_std=("pred_delta", "std"),
            n_rows=("pred_tvt", "size"),
            fold=("fold", "first"),
        ).reset_index().fillna(0.0)
        well_model_frame = base_well_features.merge(pred_well, on=["wellname", "fold"], how="left").fillna(0.0)
        model_cols = [col for col in well_model_frame.columns if col not in ["wellname", "fold", "target_bias"]]
        for fold in sorted(base_predictions["fold"].dropna().unique()):
            train_well_mask = ~well_model_frame["fold"].eq(fold)
            valid_well_mask = well_model_frame["fold"].eq(fold)
            if train_well_mask.sum() < 10 or valid_well_mask.sum() == 0:
                continue
            imputer = SimpleImputer(strategy="median")
            scaler = StandardScaler()
            X_train = imputer.fit_transform(well_model_frame.loc[train_well_mask, model_cols].replace([np.inf, -np.inf], np.nan))
            X_valid = imputer.transform(well_model_frame.loc[valid_well_mask, model_cols].replace([np.inf, -np.inf], np.nan))
            X_train = scaler.fit_transform(X_train)
            X_valid = scaler.transform(X_valid)
            y_train = well_model_frame.loc[train_well_mask, "target_bias"].to_numpy(float)
            row_mask = base_predictions["fold"].eq(fold).to_numpy()
            for alpha, bias_suffix in [(0.005, "h0005"), (0.05, "h005")]:
                bias_model = HuberRegressor(epsilon=1.35, alpha=alpha, max_iter=500)
                bias_model.fit(X_train, y_train)
                valid_bias = np.clip(bias_model.predict(X_valid), -80.0, 80.0)
                bias_by_well = dict(zip(well_model_frame.loc[valid_well_mask, "wellname"], valid_bias))
                row_bias = base_predictions.loc[row_mask, "wellname"].map(bias_by_well).fillna(0.0).to_numpy(float)
                for shrink in [0.32, 0.35, 0.38, 0.40, 0.45, 0.50]:
                    corrected_label = f"{source_label}__well_bias_{bias_suffix}_s{int(shrink * 100):02d}"
                    candidate_preds.setdefault(corrected_label, np.full(len(base_predictions), np.nan))
                    candidate_preds[corrected_label][row_mask] = source_pred[row_mask] + shrink * row_bias

    for label, pred_all in candidate_preds.items():
        for fold in sorted(base_predictions["fold"].dropna().unique()):
            valid_mask = base_predictions["fold"].eq(fold).to_numpy()
            actual = actual_all[valid_mask]
            baseline = baseline_all[valid_mask]
            pred = pred_all[valid_mask]
            metrics = metric_row(actual, pred, baseline)
            rows.append({
                "run_id": RUN_ID,
                "fold_column": fold_col,
                "fold": int(fold),
                "label": label,
                "global_model": "regime_cnn_selective_calibration",
                "regime_model": "selective_calibration",
                "schedule": label,
                "n_train_rows": int((~valid_mask).sum()),
                "n_valid_rows": int(valid_mask.sum()),
                "n_base_features": int(len(gate_cols)),
                "n_regime_features": int(sum(len(cols) for cols in SEQ_GROUPS)),
                **metrics,
            })
            tmp = base_predictions.loc[valid_mask, [
                "wellname", PRIMARY_FOLD, STRESS_FOLD, "eval_progress", "selected_analog_distance", "combined_stress_score",
                f"mix_confidence__{fold_col}", f"mix_entropy__{fold_col}", f"mix_prob_margin__{fold_col}",
                "actual_tvt", "baseline_tvt",
            ]].copy()
            tmp["fold_column"] = fold_col
            tmp["fold"] = int(fold)
            tmp["label"] = label
            tmp["global_model"] = "regime_cnn_selective_calibration"
            tmp["regime_model"] = "selective_calibration"
            tmp["schedule"] = label
            tmp["pred_tvt"] = pred
            tmp["global_pred_tvt"] = base_predictions.loc[valid_mask, "regime_pred_tvt_base"].to_numpy(float)
            tmp["regime_pred_tvt"] = base_predictions.loc[valid_mask, "cnn_pred_tvt_base"].to_numpy(float)
            tmp["blend_weight"] = np.nan
            tmp["residual"] = tmp["actual_tvt"] - tmp["pred_tvt"]
            pred_parts.append(tmp)
    return pd.DataFrame(rows), pd.concat(pred_parts, ignore_index=True)


start = time.time()
strict_base_predictions = base_candidate_predictions(PRIMARY_FOLD)
stress_base_predictions = base_candidate_predictions(STRESS_FOLD)
strict_fold_metrics, strict_predictions = add_candidate_rows(strict_base_predictions, PRIMARY_FOLD)
stress_fold_metrics, stress_predictions = add_candidate_rows(stress_base_predictions, STRESS_FOLD)
regime_summary = pd.DataFrame()
strict_diag = pd.DataFrame()
stress_diag = pd.DataFrame()
print(f"elapsed_s={time.time() - start:.1f}")
strict_fold_metrics.groupby("label")["rmse"].mean().sort_values()


C:\Users\sardo\AppData\Local\Temp\ipykernel_3664\3120776045.py:329: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  .reset_index()


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


... (72 outputs left out)

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_huber.py:343: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  self.n_iter_ = _check_optimize_result("lbfgs", opt_res, self.max_iter)


elapsed_s=777.7


label
oracle_continuous_weight                      9.964320
oracle_row_switch                            10.036860
logit_switch_floor18__well_bias_h0005_s40    11.596803
logit_switch_floor12__well_bias_h0005_s40    11.596867
logit_switch_floor15__well_bias_h0005_s40    11.597154
                                               ...    
blend_cnn_55                                 11.745292
blend_cnn_40                                 11.747829
blend_cnn_60                                 11.764215
blend_cnn_35                                 11.768445
regime_reference                             12.182093
Name: rmse, Length: 184, dtype: float64

## 7. Results And Diagnostics

In [6]:
"Summarize strict-CV, stress-CV, and regime-mean behavior."

strict_summary = (
    strict_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        rmse_mean=("rmse", "mean"),
        rmse_std=("rmse", "std"),
        mae_mean=("mae", "mean"),
        r2_mean=("r2", "mean"),
        last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        n_valid_rows=("n_valid_rows", "sum"),
        n_base_features=("n_base_features", "max"),
        n_regime_features=("n_regime_features", "max"),
    )
    .reset_index()
)
stress_summary = (
    stress_fold_metrics.groupby(["label", "global_model", "regime_model", "schedule"])
    .agg(
        stress_rmse_mean=("rmse", "mean"),
        stress_rmse_std=("rmse", "std"),
        stress_mae_mean=("mae", "mean"),
        stress_last_known_hold_rmse_mean=("last_known_hold_rmse", "mean"),
        stress_n_valid_rows=("n_valid_rows", "sum"),
    )
    .reset_index()
)
summary = strict_summary.merge(stress_summary, on=["label", "global_model", "regime_model", "schedule"], how="left")
summary["rmse_delta_vs_hold"] = summary["rmse_mean"] - summary["last_known_hold_rmse_mean"]
summary["rmse_delta_vs_mixture_prior"] = summary["rmse_mean"] - 13.443736843906953
summary["rmse_delta_vs_regime_mean_baseline"] = summary["rmse_mean"] - CONFIG["baseline_regime_mean_rmse"]
summary["rmse_delta_vs_safe_typewell_reference"] = summary["rmse_mean"] - 13.450635286744287
summary["rmse_delta_vs_public_best"] = summary["rmse_mean"] - CONFIG["current_public_best_rmse"]
summary["stress_minus_strict_rmse"] = summary["stress_rmse_mean"] - summary["rmse_mean"]
summary = summary.sort_values("rmse_mean").reset_index(drop=True)
deployable_summary = summary.loc[~summary["label"].isin(["oracle_row_switch", "oracle_continuous_weight"])].copy()
best_row = deployable_summary.iloc[0].copy()
best_label = str(best_row["label"])

predictions = pd.concat([strict_predictions, stress_predictions], ignore_index=True)
predictions["progress_zone"] = pd.cut(predictions["eval_progress"], bins=[-0.001, 0.25, 0.50, 0.75, 1.001], labels=["z0", "z1", "z2", "z3"])
predictions["analog_distance_bin"] = pd.qcut(predictions["selected_analog_distance"], q=5, labels=["closest", "near", "middle", "far", "farthest"], duplicates="drop")

zone_summary = (
    predictions.groupby(["fold_column", "label", "progress_zone"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
        "mean_mix_confidence": float(group.filter(like="mix_confidence").iloc[:, 0].mean()),
    }), include_groups=False)
    .reset_index()
)
zone_summary["rmse_delta_vs_global"] = zone_summary["rmse"] - zone_summary["global_rmse"]

analog_bin_summary = (
    predictions.groupby(["fold_column", "label", "analog_distance_bin"], observed=True)
    .apply(lambda group: pd.Series({
        "n_rows": len(group),
        "n_wells": group["wellname"].nunique(),
        "rmse": rmse(group["actual_tvt"], group["pred_tvt"]),
        "global_rmse": rmse(group["actual_tvt"], group["global_pred_tvt"]),
        "regime_rmse": rmse(group["actual_tvt"], group["regime_pred_tvt"]),
        "baseline_rmse": rmse(group["actual_tvt"], group["baseline_tvt"]),
        "mean_blend_weight": float(group["blend_weight"].mean()),
    }), include_groups=False)
    .reset_index()
)
analog_bin_summary["rmse_delta_vs_global"] = analog_bin_summary["rmse"] - analog_bin_summary["global_rmse"]

well_diagnostics = (
    predictions.loc[(predictions["fold_column"].eq(PRIMARY_FOLD)) & (predictions["label"].eq(best_label))]
    .groupby("wellname")
    .agg(
        n_rows=("actual_tvt", "size"),
        strict_fold=(PRIMARY_FOLD, "first"),
        analog_fold=(STRESS_FOLD, "first"),
        mean_selected_analog_distance=("selected_analog_distance", "mean"),
        mean_blend_weight=("blend_weight", "mean"),
        mean_residual=("residual", "mean"),
        mean_abs_residual=("residual", lambda series: float(np.abs(series).mean())),
        rmse=("residual", lambda series: float(np.sqrt(np.mean(np.square(series))))),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

regime_diagnostics = pd.concat([strict_diag, stress_diag], ignore_index=True)
if regime_diagnostics.empty:
    regime_summary = pd.DataFrame(columns=[
        "fold_column", "label", "global_model", "regime_model", "schedule",
        "mean_blend_weight", "max_blend_weight", "train_global_rmse", "train_regime_rmse",
    ])
else:
    regime_summary = (
        regime_diagnostics.groupby(["fold_column", "label", "global_model", "regime_model", "schedule"])
        .agg(
            mean_blend_weight=("mean_blend_weight", "mean"),
            max_blend_weight=("max_blend_weight", "max"),
            train_global_rmse=("train_global_rmse", "mean"),
            train_regime_rmse=("train_regime_rmse", "mean"),
        )
        .reset_index()
    )

print(summary.head(18).to_string(index=False))

                                    label                     global_model          regime_model                                  schedule  rmse_mean  rmse_std  mae_mean  r2_mean  last_known_hold_rmse_mean  n_valid_rows  n_base_features  n_regime_features  stress_rmse_mean  stress_rmse_std  stress_mae_mean  stress_last_known_hold_rmse_mean  stress_n_valid_rows  rmse_delta_vs_hold  rmse_delta_vs_mixture_prior  rmse_delta_vs_regime_mean_baseline  rmse_delta_vs_safe_typewell_reference  rmse_delta_vs_public_best  stress_minus_strict_rmse
                 oracle_continuous_weight regime_cnn_selective_calibration selective_calibration                  oracle_continuous_weight   9.964320  0.592871  6.207468 0.999737                  15.571277         92760              149                 51         10.211658         0.295270         6.259270                         15.581239                92760           -5.606957                    -3.479416                           -3.128463             

## 8. Artifact Export

In [7]:
"Export artifacts, report, and model-comparison row."

best_strict_rmse = float(best_row["rmse_mean"])
best_stress_rmse = float(best_row["stress_rmse_mean"])
best_global = deployable_summary.sort_values("rmse_mean").iloc[0]
best_confidence = summary.loc[summary["label"].isin(["oracle_row_switch", "oracle_continuous_weight"])].sort_values("rmse_mean").iloc[0]

if best_strict_rmse < 11.612923027271995 and best_stress_rmse <= 11.920813188903125:
    recommendation = "package_cnn_regime_selective_calibration"
elif best_strict_rmse < 11.612923027271995:
    recommendation = "strict_improves_stress_regresses"
elif best_stress_rmse <= 11.920813188903125:
    recommendation = "stress_improves_strict_regresses"
elif best_strict_rmse < CONFIG["baseline_regime_mean_rmse"]:
    recommendation = "diagnostic_only_current_best_worse"
else:
    recommendation = "diagnostic_only"

findings = pd.DataFrame([
    {"finding": "best_overall", "detail": best_label, "value": best_strict_rmse, "interpretation": f"Best strict RMSE with analog stress RMSE {best_stress_rmse:.4f}."},
    {"finding": "best_weight_candidate", "detail": str(best_global["label"]), "value": float(best_global["rmse_mean"]), "interpretation": "Best deployable selective-calibration/static gate candidate."},
    {"finding": "oracle_upper_bound", "detail": str(best_confidence["label"]), "value": float(best_confidence["rmse_mean"]), "interpretation": "Oracle row switch upper bound is leakage-only and not deployable."},
    {"finding": "decision", "detail": recommendation, "value": best_strict_rmse, "interpretation": "Promotion rule checks the direct regime-mean anchor, analog-distance stress behavior, and current public context."},
])

paths = {
    "summary": TABLE_DIR / "model__cnn_regime_selective_calibration_summary__2026-07-01.csv",
    "fold_metrics": TABLE_DIR / "model__cnn_regime_selective_calibration_fold_metrics__2026-07-01.csv",
    "regime_diagnostics": TABLE_DIR / "model__cnn_regime_selective_calibration_regime_diagnostics__2026-07-01.csv",
    "zone_summary": TABLE_DIR / "model__cnn_regime_selective_calibration_zone_summary__2026-07-01.csv",
    "analog_bin_summary": TABLE_DIR / "model__cnn_regime_selective_calibration_analog_bin_summary__2026-07-01.csv",
    "well_diagnostics": TABLE_DIR / "model__cnn_regime_selective_calibration_well_diagnostics__2026-07-01.csv",
    "findings": TABLE_DIR / "model__cnn_regime_selective_calibration_findings__2026-07-01.csv",
}
exports = {
    "summary": summary,
    "fold_metrics": pd.concat([strict_fold_metrics, stress_fold_metrics], ignore_index=True),
    "regime_diagnostics": regime_summary,
    "zone_summary": zone_summary,
    "analog_bin_summary": analog_bin_summary,
    "well_diagnostics": well_diagnostics,
    "findings": findings,
}
for key, frame in exports.items():
    frame.to_csv(paths[key], index=False)
    shutil.copy2(paths[key], RUN_DIR / paths[key].name)

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path) if comparison_path.exists() else pd.DataFrame()
new_row = pd.DataFrame([{
    "run_id": RUN_ID,
    "model_family": "neural_network",
    "model_name": "cnn_regime_selective_calibration",
    "best_candidate": best_label,
    "local_cv_rmse": best_strict_rmse,
    "local_cv_mae": float(best_row["mae_mean"]),
    "public_lb_rmse": np.nan,
    "status": recommendation,
    "notes": f"stress_rmse={best_stress_rmse:.4f}; best_confidence={float(best_confidence['rmse_mean']):.4f}",
    "updated_at": RUN_DATE,
}])
if not comparison.empty and "run_id" in comparison.columns:
    comparison = comparison.loc[comparison["run_id"] != RUN_ID].copy()
comparison = pd.concat([comparison, new_row], ignore_index=True, sort=False)
comparison.to_csv(comparison_path, index=False)
shutil.copy2(comparison_path, RUN_DIR / comparison_path.name)

report_path = REPORT_DIR / "model__cnn_regime_selective_calibration__2026-07-01.md"
report = f"""# CNN-Regime Selective Trajectory Calibration

## Result

Best overall candidate: `{best_label}` with strict-CV RMSE {best_strict_rmse:.4f}, MAE {float(best_row["mae_mean"]):.4f}, and analog-distance stress RMSE {best_stress_rmse:.4f}.

Oracle row-switch upper bound: `{best_confidence["label"]}` with RMSE {float(best_confidence["rmse_mean"]):.4f}. This is leakage-only and is not deployable, but it shows the CNN and regime anchor have useful complementary errors.

## Model Structure

The audit recomputes the direct pseudo-top regime-mean anchor and compact local-window CNN specialist with hidden-test-safe well-trajectory context features. It then tests uniform CNN blends, simple stress/confidence/disagreement gates, conservative well-bias corrections, and a leakage-only oracle row switch.

## Leakage Notes

Direct formation columns are used only as fold-local labels for pseudo-top proxy training. Regime probabilities, regime means, CNN windows, disagreement features, and gates use held-out pseudo-top predictions and hidden-test-safe features. The oracle row switch uses validation targets and is diagnostic only.

## Decision

Recommendation: `{recommendation}`.

## Next Move

Selective trajectory calibration tests whether wider logit floors, switch powers, and conservative well-bias shrinkage can recover more of the new oracle gap.
"""
report_path.write_text(report, encoding="utf-8")
shutil.copy2(report_path, RUN_DIR / report_path.name)

print(findings.to_string(index=False))
print(report_path)

              finding                                    detail     value                                                                                                    interpretation
         best_overall logit_switch_floor18__well_bias_h0005_s40 11.596803                                                                 Best strict RMSE with analog stress RMSE 11.9062.
best_weight_candidate logit_switch_floor18__well_bias_h0005_s40 11.596803                                                      Best deployable selective-calibration/static gate candidate.
   oracle_upper_bound                  oracle_continuous_weight  9.964320                                                 Oracle row switch upper bound is leakage-only and not deployable.
             decision  package_cnn_regime_selective_calibration 11.596803 Promotion rule checks the direct regime-mean anchor, analog-distance stress behavior, and current public context.
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tv

## 9. Findings

In [8]:
"Show headline result tables."

display(summary.head(15))
display(regime_summary.loc[regime_summary["label"].eq(best_label)].head())
display(findings)

,label,global_model,regime_model,schedule,rmse_mean,rmse_std,mae_mean,r2_mean,last_known_hold_rmse_mean,n_valid_rows,n_base_features,n_regime_features,stress_rmse_mean,stress_rmse_std,stress_mae_mean,stress_last_known_hold_rmse_mean,stress_n_valid_rows,rmse_delta_vs_hold,rmse_delta_vs_mixture_prior,rmse_delta_vs_regime_mean_baseline,rmse_delta_vs_safe_typewell_reference,rmse_delta_vs_public_best,stress_minus_strict_rmse
0,oracle_continuous_weight,regime_cnn_selective_calibration,selective_calibration,oracle_continuous_weight,9.964320,0.592871,6.207468,0.999737,15.571277,92760,149,51,10.211658,0.295270,6.259270,15.581239,92760,-5.606957,-3.479416,-3.128463,-3.486315,-4.507680,0.247337
1,oracle_row_switch,regime_cnn_selective_calibration,selective_calibration,oracle_row_switch,10.036860,0.574480,6.600274,0.999734,15.571277,92760,149,51,10.305002,0.291818,6.724583,15.581239,92760,-5.534417,-3.406877,-3.055924,-3.413775,-4.435140,0.268142
2,logit_switch_floor18__well_bias_h0005_s40,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor18__well_bias_h0005_s40,11.596803,0.534843,8.256387,0.999645,15.571277,92760,149,51,11.906193,0.522928,8.464863,15.581239,92760,-3.974474,-1.846933,-1.495980,-1.853832,-2.875197,0.309389
3,logit_switch_floor12__well_bias_h0005_s40,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor12__well_bias_h0005_s40,11.596867,0.539879,8.250032,0.999645,15.571277,92760,149,51,11.905872,0.531227,8.454209,15.581239,92760,-3.974410,-1.846869,-1.495916,-1.853768,-2.875133,0.309005
4,logit_switch_floor15__well_bias_h0005_s40,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor15__well_bias_h0005_s40,11.597154,0.538467,8.253026,0.999645,15.571277,92760,149,51,11.903843,0.524132,8.457271,15.581239,92760,-3.974123,-1.846583,-1.495629,-1.853481,-2.874846,0.306689
5,logit_switch_floor10__well_bias_h0005_s40,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor10__well_bias_h0005_s40,11.597264,0.540802,8.248213,0.999645,15.571277,92760,149,51,11.906099,0.532999,8.451453,15.581239,92760,-3.974013,-1.846472,-1.495519,-1.853371,-2.874736,0.308834
6,logit_switch_floor18__well_bias_h0005_s45,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor18__well_bias_h0005_s45,11.597267,0.538620,8.260453,0.999645,15.571277,92760,149,51,11.908430,0.518205,8.475349,15.581239,92760,-3.974010,-1.846470,-1.495517,-1.853369,-2.874733,0.311163
7,logit_switch_floor12__well_bias_h0005_s38,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor12__well_bias_h0005_s38,11.597459,0.538526,8.249249,0.999645,15.571277,92760,149,51,11.905754,0.532919,8.450784,15.581239,92760,-3.973818,-1.846277,-1.495324,-1.853176,-2.874541,0.308294
8,logit_switch_floor18__well_bias_h0005_s38,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor18__well_bias_h0005_s38,11.597693,0.533286,8.255974,0.999645,15.571277,92760,149,51,11.906531,0.524881,8.461734,15.581239,92760,-3.973583,-1.846043,-1.495090,-1.852942,-2.874307,0.308838
9,logit_switch_floor10__well_bias_h0005_s38,regime_cnn_selective_calibration,selective_calibration,logit_switch_floor10__well_bias_h0005_s38,11.597769,0.539572,8.247259,0.999645,15.571277,92760,149,51,11.905955,0.534792,8.447959,15.581239,92760,-3.973508,-1.845968,-1.495015,-1.852867,-2.874231,0.308187


,fold_column,label,global_model,regime_model,schedule,mean_blend_weight,max_blend_weight,train_global_rmse,train_regime_rmse


,finding,detail,value,interpretation
0,best_overall,logit_switch_floor18__well_bias_h0005_s40,11.596803,Best strict RMSE with analog stress RMSE 11.9062.
1,best_weight_candidate,logit_switch_floor18__well_bias_h0005_s40,11.596803,Best deployable selective-calibration/static g...
2,oracle_upper_bound,oracle_continuous_weight,9.964320,Oracle row switch upper bound is leakage-only ...
3,decision,package_cnn_regime_selective_calibration,11.596803,Promotion rule checks the direct regime-mean a...
